# TEAM 04 — Ames Housing: Does Adding More Features Improve Prediction?
## An Investigation of Informative, Irrelevant, and Redundant Features

**Research Question:** Does adding more features necessarily improve prediction?

**Required structure:** Question → Hypothesis → Experiment → Evidence → Analysis → Conclusion

---

## Team Contributions

| Member | Area | Source file |
|---|---|---|
| Member 1 | Experimental Design & Feature Construction | `src/feature_sets.py` |
| Member 2 | Data Processing & Preprocessing | `src/data.py`, `src/preprocessing.py` |
| Member 3 | Model Behaviour | `src/models.py`, `src/experiments.py` |
| Member 4 | Validation, Metrics & Statistical Analysis | `src/validation.py`, `src/analysis.py` |


## 0. Imports and Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)


---
# MEMBER 1 — Experimental Design & Feature Construction
**File:** `src/feature_sets.py`  
**Branch:** `feature/member1-feature-design`

## Hypotheses (defined before modeling)

| Hypothesis | Experiment | Expected direction |
|---|---|---|
| H1 — Meaningful features improve performance | M10 → M20 | RMSE down |
| H2 — Irrelevant features do NOT improve | M20 → M20+I | RMSE up or flat |
| H3 — Redundant features add no new information | M20 → M20+R | RMSE flat |
| H4 — Combined exposes model-specific sensitivity | M20 → M20+I+R | Model-dependent |


### 1.1 Feature lists

In [ ]:
from src.feature_sets import (
    M10_FEATURES, M20_FEATURES,
    IRRELEVANT_FEATURE_NAMES, REDUNDANT_FEATURE_NAMES,
    REDUNDANT_SPECS,
    add_irrelevant_features, add_redundant_features,
    build_feature_configs, redundancy_correlation_table,
)

print(f"M10 features ({len(M10_FEATURES)}):")
for f in M10_FEATURES:
    print(f"  {f}")

print(f"\nM20 features ({len(M20_FEATURES)}):")
for f in M20_FEATURES:
    mark = "  " if f in M10_FEATURES else "+ "
    print(f"  {mark}{f}")


### 1.2 Experiment configuration table

In [ ]:
config_doc = {
    "M10":     ("10 meaningful",               "Secondary: H1 meaningful gain"),
    "M20":     ("20 meaningful",               "PRIMARY BASELINE for H2/H3/H4"),
    "M20_I":   ("20 meaningful + 10 irrel.",   "Main experiment: H2"),
    "M20_R":   ("20 meaningful + 5 redund.",   "Main experiment: H3"),
    "M20_I_R": ("20 meaningful + both",        "Main experiment: H4"),
    "ALL":     ("All original features",       "Contextual only — NOT main baseline"),
}
pd.DataFrame.from_dict(
    config_doc, orient="index",
    columns=["Contents", "Role"]
)


---
# MEMBER 2 — Data Processing & Preprocessing
**Files:** `src/data.py`, `src/preprocessing.py`  
**Branch:** `feature/member2-preprocessing`


### 2.1 Load and inspect data

In [ ]:
from src.data import load_data, inspect_data, recode_absence_nans, prepare_XY, make_train_test_split

df_raw = load_data("data/raw/train.csv")
inspect_data(df_raw)


### 2.2 Recode absence NaNs (Ames-specific semantics)

> In the Ames dataset, NaN does not always mean unknown.
> For garage columns, NaN means the house has no garage.
> Categorical absence -> "None" | Numerical absence -> 0


In [ ]:
df = recode_absence_nans(df_raw)

missing_after = df.isnull().sum()
missing_after = missing_after[missing_after > 0].sort_values(ascending=False)
print("Remaining genuine unknowns after absence recoding:")
display(missing_after.head(20).to_frame("RemainingMissing"))


### 2.3 Separate predictors / remove identifiers

In [ ]:
X, y = prepare_XY(df)

# Confirm that Order, PID, Id, SalePrice are NOT in X
for col in ["Order", "PID", "Id", "SalePrice"]:
    print(f"  {col} in X: {col in X.columns}")


### 2.4 Fixed 80/20 train/test split

In [ ]:
X_train, X_test, y_train, y_test = make_train_test_split(X, y)


### 2.5 Add synthetic features to full dataset

In [ ]:
X_experiment = add_irrelevant_features(X)
X_experiment = add_redundant_features(X_experiment)

print(f"X shape before synthetic: {X.shape}")
print(f"X shape after synthetic:  {X_experiment.shape}")


### 2.6 Verify redundant feature construction

In [ ]:
redund_corr_table = redundancy_correlation_table(X_experiment)
print("Redundant feature Pearson correlations with originals:")
display(redund_corr_table)


### 2.7 Preprocessing pipeline design

| Step | Numerical | Categorical |
|---|---|---|
| Imputation | Median (genuine unknowns) | Most-frequent |
| Scaling | StandardScaler (Ridge only) | No scaling |
| Encoding | No encoding | OneHotEncoder |

Pipeline enforces that all statistics are learned from training folds only — no leakage.


---
# MEMBER 3 — Model Behaviour
**Files:** `src/models.py`, `src/experiments.py`  
**Branch:** `feature/member3-models`


### 3.1 Model configurations

**Ridge(alpha=10):** L2 regularized linear model.
Sensitive to multicollinearity; requires StandardScaler.

**RandomForestRegressor(n_estimators=300, max_features="sqrt"):**
Non-linear tree ensemble. Scale-invariant.
max_features="sqrt" means each split considers sqrt(n_features) candidates.
Code and explanation are consistent (handoff spec section 18).


In [ ]:
from src.models import make_models

models = make_models()
for name, m in models.items():
    print(f"{name}: {m}")


### 3.2 Build feature configurations

In [ ]:
all_original_cols = list(X.columns)
feature_configs = build_feature_configs(all_original_cols)

config_summary = pd.DataFrame([
    {"Config": name, "Features": len(feats)}
    for name, feats in feature_configs.items()
])
display(config_summary)


### 3.3 Run experiments on fixed train/test split

In [ ]:
from src.experiments import run_all_experiments, compute_pairwise_deltas

print("Running all configurations x models on fixed split...")
results_df = run_all_experiments(
    X_experiment, X_train, X_test, y_train, y_test, all_original_cols
)

display(
    results_df[
        ["Config","Model","Raw_Features","Train_RMSE","Test_RMSE","Test_MAE","Test_R2","Generalization_Gap"]
    ].sort_values(["Model","Config"]).round(2)
)


### 3.4 Pairwise delta analysis (single split)

In [ ]:
delta_df = compute_pairwise_deltas(results_df)

print("Pairwise delta RMSE (positive = worse, negative = better):")
display(delta_df.round(2))


---
# MEMBER 4 — Validation, Metrics & Statistical Analysis
**Files:** `src/validation.py`, `src/analysis.py`  
**Branch:** `feature/member4-validation`


### 4.1 Cross-validation design

**5-fold CV using the SAME KFold object for every configuration.**

The same folds enable a paired comparison: delta RMSE between M20 and M20+I
on fold k is attributable to the feature change, not random fold variation.

The final test set (20%) is never used during CV.


### 4.2 Run 5-fold cross-validation

In [ ]:
from src.validation import run_cross_validation, compute_cv_deltas, cv_summary_table

print("Running 5-fold CV for all configurations x models...")
print("(This may take 1-2 minutes for Random Forest)")

cv_df = run_cross_validation(X_experiment, X_train, y_train, all_original_cols)


### 4.3 CV results: mean plus/minus SD

In [ ]:
cv_summary = cv_summary_table(cv_df)

print("Cross-Validation Summary (mean +/- SD across 5 folds):")
display(cv_summary.sort_values(["Model","Config"]).round(2))


### 4.4 CV pairwise deltas with 95% confidence intervals

In [ ]:
cv_delta_df = compute_cv_deltas(cv_df)

print("CV Pairwise delta RMSE with 95% CI:")
print("If CI does not cross 0, effect is consistent across folds.")
display(cv_delta_df)


### 4.5 Plot 1 — RMSE by feature configuration

In [ ]:
from src.analysis import (
    plot_rmse_by_config, plot_generalization,
    plot_cv_distribution, plot_delta_rmse,
    plot_feature_count_vs_rmse, plot_redundancy_heatmap,
    build_evidence_table,
)

plot_rmse_by_config(results_df)


### 4.6 Plot 2 — Train vs CV vs Test RMSE (generalization)

In [ ]:
plot_generalization(results_df, cv_df)


### 4.7 Plot 3 — CV RMSE distribution across folds

In [ ]:
plot_cv_distribution(cv_df)


### 4.8 Plot 4 — Pairwise delta RMSE

In [ ]:
plot_delta_rmse(cv_delta_df)


### 4.9 Plot 5 — Feature count vs Test RMSE

In [ ]:
plot_feature_count_vs_rmse(results_df)


### 4.10 Redundancy heatmap

In [ ]:
plot_redundancy_heatmap(X_experiment, REDUNDANT_SPECS)


### 4.11 Save final evidence tables

In [ ]:
evidence_table = build_evidence_table(results_df, cv_df, delta_df, cv_delta_df)
print("Evidence table saved.")
display(evidence_table.round(2))


---
# Analysis and Conclusions

> Complete this section after running the notebook and obtaining results.

## H1 — Did adding meaningful features (M10 to M20) improve performance?
*(Report delta RMSE, interpret direction)*

## H2 — Did adding irrelevant features (M20 to M20+I) hurt or help?
*(Compare Ridge vs RF reaction)*

## H3 — Did redundant features (M20 to M20+R) add information?
*(Near-zero delta expected)*

## H4 — Did the combined set (M20 to M20+I+R) behave differently?
*(Check for additive vs non-additive effects)*

## Conclusion

> Under the controlled experimental conditions, ...
> (Complete based on evidence — avoid absolute claims)

## Limitations

1. Results are specific to the Ames dataset.
2. Synthetic redundant features represent noisy linear transforms — one form of redundancy.
3. M20 feature set is an experimental definition chosen by domain knowledge.
4. Fixed hyperparameters may affect observed model behaviour.
5. Conclusions about this controlled experiment do not generalise to all datasets.
